In [ ]:
import os
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
import io
import msoffcrypto
import pandas as pd

gauth = GoogleAuth()
gauth.LocalWebserverAuth()
drive = GoogleDrive(gauth)

FILES = [
    {"id": "1gLprbmhabu5u-udI2xgKtHuM4XYl29gu", "label": "Incidents"},
    {"id": "1XZ7_PRw6dBtv-qfScNsCVgCeJzQuSsiH", "label": "Meaningful Activities"},
    {"id": "1pggGCCCOP4YPZNhi0xym3LYqi6B5-6LS", "label": "Medication"},
]

PASSWORD = os.environ["DATASET_PASSWORD"] 

all_results = []
datasets = {}

for entry in FILES:
    file_id = entry["id"]
    label = entry["label"]
    print(f"\n=== Processing {label} ===")

    file = drive.CreateFile({'id': file_id})
    file.FetchMetadata()
    stream = file.GetContentIOBuffer()
    encrypted_bytes = io.BytesIO(stream.read())
    print(f"Downloaded {encrypted_bytes.getbuffer().nbytes:,} bytes")

    decrypted = io.BytesIO()
    office_file = msoffcrypto.OfficeFile(encrypted_bytes)
    office_file.load_key(password=PASSWORD)
    office_file.decrypt(decrypted)
    decrypted.seek(0)
    df = pd.read_excel(decrypted, sheet_name=0, engine="openpyxl")
    print(f"{label}: {df.shape[0]} rows × {df.shape[1]} cols")

    datasets[label] = df


    # for sheet, df in dfs.items():
    #     print(f"  Sheet {sheet}: {df.shape[0]} rows, {df.shape[1]} cols")

    # all_results.append((label, dfs))
df_a = datasets["Incidents"]
df_b = datasets["Meaningful Activities"]
df_c = datasets["Medication"]
print("\nall files processed")


In [ ]:
df_a.head()

In [ ]:
df_b.head()

In [ ]:
df_c.head()

In [ ]:
df_a.loc[(df_a['Seclusion'] == 1) & (df_a['Restraint'] == 1), 'Restraint_SubType'] = "Physical Aggression"


In [ ]:
df = df_a

<p align="center">
  <span style="font-size:3em; font-weight:bold;">Medication Merger</span>
</p>

In [ ]:
# meds = df_c.copy()
# df = df.copy()  

# meds['Prescription_Start_Date'] = pd.to_datetime(meds['Prescription_Start_Date'])
# meds['Prescription_End_Date'] = pd.to_datetime(meds['Prescription_End_Date'])
# df['Time_Stamp'] = pd.to_datetime(df['Time_Stamp'])


# meds['is_antipsychotic'] = meds['Prescription_Description'].str.contains('antipsychotic', case=False, na=False).astype(int)
# meds['is_prn'] = meds['Prescription_Type'].str.contains('PRN', case=False, na=False).astype(int)

# antipsy = meds[meds['is_antipsychotic'] == 1].copy()
# antipsy['Prescription_End_Date'] = antipsy['Prescription_End_Date'].fillna(pd.Timestamp.max)

# antipsy_intervals = pd.IntervalIndex.from_arrays(
#     antipsy['Prescription_Start_Date'],
#     antipsy['Prescription_End_Date'],
#     closed='both'
# )

# def was_on_antipsychotic(events):
#     result = []
#     for t in events['Time_Stamp']:
#         result.append(antipsy_intervals.contains(t).any())
#     return pd.Series(result, index=events.index)

# df['on_antipsychotic_24h'] = was_on_antipsychotic(df).astype(int)


# prn = meds[meds['is_prn'] == 1][['Patient_ID','Prescription_Start_Date']].copy()
# prn = prn.sort_values(['Patient_ID','Prescription_Start_Date'])
# df = df.sort_values(['Patient_ID','Time_Stamp'])

# df['num_prn_72h'] = 0
# for pid, group in prn.groupby('Patient_ID'):
#     mask = df['Patient_ID'] == pid
#     df.loc[mask, 'num_prn_72h'] = df.loc[mask].apply(
#         lambda row: ((group['Prescription_Start_Date'] >= row['Time_Stamp'] - pd.Timedelta(hours=72)) &
#                      (group['Prescription_Start_Date'] <= row['Time_Stamp'])).sum(),
#         axis=1
#     )

# print(df[['Patient_ID','Time_Stamp','on_antipsychotic_24h','num_prn_72h']].head())


<p align="center">
  <span style="font-size:3em; font-weight:bold;">Medication & MA Merger</span>
</p>

In [ ]:
import pandas as pd
import numpy as np


df = df_a.copy()             
meds = df_c.copy()        
activities = df_b.copy()  

df['Time_Stamp'] = pd.to_datetime(df['Time_Stamp'])
meds['Prescription_Start_Date'] = pd.to_datetime(meds['Prescription_Start_Date'])
meds['Prescription_End_Date'] = pd.to_datetime(meds['Prescription_End_Date'])
activities['Appointment_Date'] = pd.to_datetime(activities['Appointment_Date'])

# MEDICATION FEATURES

meds['is_antipsychotic'] = meds['Prescription_Description'].str.contains('antipsychotic', case=False, na=False).astype(int)
meds['is_prn'] = meds['Prescription_Type'].str.contains('PRN', case=False, na=False).astype(int)

antipsy = meds[meds['is_antipsychotic'] == 1].copy()
antipsy['Prescription_End_Date'] = antipsy['Prescription_End_Date'].fillna(pd.Timestamp.max)

antipsy_intervals = pd.IntervalIndex.from_arrays(
    antipsy['Prescription_Start_Date'],
    antipsy['Prescription_End_Date'],
    closed='both'
)

def was_on_antipsychotic(events):
    result = []
    for t in events['Time_Stamp']:
        result.append(antipsy_intervals.contains(t).any())
    return pd.Series(result, index=events.index)

df['on_antipsychotic_24h'] = was_on_antipsychotic(df).astype(int)


prn = meds[meds['is_prn'] == 1][['Patient_ID','Prescription_Start_Date']].copy()
prn = prn.sort_values(['Patient_ID','Prescription_Start_Date'])
df = df.sort_values(['Patient_ID','Time_Stamp'])

df['num_prn_72h'] = 0
for pid, group in prn.groupby('Patient_ID'):
    mask = df['Patient_ID'] == pid
    df.loc[mask, 'num_prn_72h'] = df.loc[mask].apply(
        lambda row: ((group['Prescription_Start_Date'] >= row['Time_Stamp'] - pd.Timedelta(hours=72)) &
                     (group['Prescription_Start_Date'] <= row['Time_Stamp'])).sum(),
        axis=1
    )


df['activity_24h'] = 0
df['time_since_last_activity'] = np.nan
df['OT_flag'] = 0
df['Section17_flag'] = 0


for pid, group in activities.groupby('Patient_ID'):
    mask = df['Patient_ID'] == pid
    for idx, row in df.loc[mask].iterrows():
        past = group[group['Appointment_Date'] <= row['Time_Stamp']]
        if past.empty:
            continue
        last_time = past['Appointment_Date'].max()
        hours_since = (row['Time_Stamp'] - last_time).total_seconds() / 3600
        df.at[idx, 'time_since_last_activity'] = hours_since
        df.at[idx, 'activity_24h'] = int(hours_since <= 24)
        df.at[idx, 'OT_flag'] = int((past['Appointment_Name'].str.contains('OT', case=False)).any())
        df.at[idx, 'Section17_flag'] = int((past['Appointment_Name'].str.contains('Section 17', case=False)).any())

df['activity_data_available'] = df['time_since_last_activity'].notna().astype(int)

print(df[['Patient_ID','Time_Stamp','on_antipsychotic_24h','num_prn_72h',
          'activity_24h','time_since_last_activity','OT_flag','Section17_flag']].head())


In [ ]:
df.shape

<p align="center">
  This Pulls the Physical Aggression marker out of the Restraint subtype
</p>

In [ ]:
df['restraint_physical_aggression'] = (
    df['Restraint_SubType'] == 'Physical Aggression'
).astype('int8')


In [ ]:
(df['restraint_physical_aggression'] == 1).sum()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.shape

In [ ]:
len(df['Patient_ID'].unique())

<center>There are 204'780 entries from 2957 patients within this dataset. 

In [ ]:
df.isnull().sum()

Fix NaN Values and create binary features for RNN

In [ ]:
df['MH_Section'] = df['MH_Section'].fillna('Unknown')

df['Restraint_SubType'] = df['Restraint_SubType'].fillna('None')
df['Death_SubType'] = df['Death_SubType'].fillna('None')
df['SelfHarm_SubTypeCategory'] = df['SelfHarm_SubTypeCategory'].fillna('None')
df['Safeguarding_Type'] = df['Safeguarding_Type'].fillna('None')
df['On_Obs_At_Event'] = df['On_Obs_At_Event'].fillna('Unknown')

df['any_restraint'] = (df['Restraint_SubType'] != 'None').astype('int8')
# df['any_death'] = (df['Death_SubType'] != 'None').astype('int8')
df['any_selfharm'] = (df['SelfHarm_SubTypeCategory'] != 'None').astype('int8')
df['any_safeguarding'] = (df['Safeguarding_Type'] != 'None').astype('int8')
df['any_obs_event'] = (df['On_Obs_At_Event'] != 'Unknown').astype('int8')

df['restraint_physical_aggression'] = (
    df['Restraint_SubType'] == 'Physical Aggression'
).astype('int8')

print(df[['MH_Section', 'Restraint_SubType', 'Death_SubType', 
          'SelfHarm_SubTypeCategory', 'Safeguarding_Type', 
          'On_Obs_At_Event', 
          'any_restraint', 'any_death', 'any_selfharm', 
          'any_safeguarding', 'any_obs_event', 
          'restraint_physical_aggression']].head())


In [ ]:
df.isnull().sum()

In [ ]:
df.drop(columns=['any_death'])

In [ ]:
import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf 

<p align="center">
  <span style="font-size:3em; font-weight:bold;">Time Series Formatting</span>
</p>

In [ ]:
import pandas as pd
import numpy as np
from keras.preprocessing.sequence import pad_sequences

df_a = df_a.copy()
df_b = df_b.copy()
df_c = df_c.copy()

df_a['Time_Stamp'] = pd.to_datetime(df_a['Time_Stamp'])
df_b['Appointment_Date'] = pd.to_datetime(df_b['Appointment_Date'])
df_c['Prescription_Start_Date'] = pd.to_datetime(df_c['Prescription_Start_Date'])
df_c['Prescription_End_Date'] = pd.to_datetime(df_c['Prescription_End_Date'])

df_a['Day'] = df_a['Time_Stamp'].dt.floor('D')
df_b['Day'] = df_b['Appointment_Date'].dt.floor('D')
df_c['start_day'] = df_c['Prescription_Start_Date'].dt.floor('D')
df_c['end_day'] = df_c['Prescription_End_Date'].dt.floor('D')

span = df_a.groupby('Patient_ID')['Day'].agg(['min','max']).reset_index()
ranges = span.apply(lambda r: pd.date_range(r['min'], r['max'], freq='D'), axis=1)
timeline = pd.DataFrame({
    'Patient_ID': np.repeat(span['Patient_ID'].to_numpy(), ranges.map(len).to_numpy()),
    'Day': np.concatenate([rng.to_numpy() for rng in ranges])
})
timeline.sort_values(['Patient_ID','Day'], inplace=True, ignore_index=True)

# Aggression today
aggr_today = (df_a.loc[df_a['Restraint_SubType'].eq('Physical Aggression'),
                       ['Patient_ID','Day']]
              .assign(aggression_today=1)
              .drop_duplicates())
timeline = timeline.merge(aggr_today, on=['Patient_ID','Day'], how='left')
timeline['aggression_today'] = timeline['aggression_today'].fillna(0).astype('int8')

# Flags
flags = df_a[['Patient_ID','Day']].copy()
flags['any_restraint'] = (df_a['Restraint_SubType'] != 'None').astype('int8')
flags['any_selfharm'] = (df_a['SelfHarm_SubTypeCategory'] != 'None').astype('int8')
flags['any_safeguarding'] = (df_a['Safeguarding_Type'] != 'None').astype('int8')
flags['any_obs_event'] = (df_a['On_Obs_At_Event'] != 'Unknown').astype('int8')
flags['restraint_physical_aggression'] = (df_a['Restraint_SubType'] == 'Physical Aggression').astype('int8')

flags = flags.groupby(['Patient_ID','Day'], as_index=False).max()
timeline = timeline.merge(flags, on=['Patient_ID','Day'], how='left')

binary_cols = [
    'any_restraint',
    'any_selfharm',
    'any_safeguarding',
    'any_obs_event',
    'restraint_physical_aggression'
]
timeline[binary_cols] = timeline[binary_cols].fillna(0).astype('int8')

# Activity today
act_today = (df_b[['Patient_ID','Day']]
             .assign(activity_today=1)
             .drop_duplicates())
timeline = timeline.merge(act_today, on=['Patient_ID','Day'], how='left')
timeline['activity_today'] = timeline['activity_today'].fillna(0).astype('int8')

timeline['last_act_date'] = timeline['Day'].where(timeline['activity_today'].eq(1))
timeline['last_act_date'] = timeline.groupby('Patient_ID')['last_act_date'].ffill()

timeline['time_since_last_activity_hours'] = (
    (timeline['Day'] - timeline['last_act_date']) / pd.Timedelta(hours=1)
).astype('float32')
timeline['time_since_last_activity_hours'] = timeline['time_since_last_activity_hours'].fillna(-1)
timeline['activity_data_available'] = timeline['last_act_date'].notna().astype('int8')

# Antipsychotic prescriptions
is_anti = df_c['Prescription_Description'].str.contains('antipsychotic', case=False, na=False)
anti = df_c.loc[is_anti, ['Patient_ID','start_day','end_day']].copy()
anti['end_day'] = anti['end_day'].fillna(anti['start_day'] + pd.Timedelta(days=3650))
anti['end_day_plus1'] = anti['end_day'] + pd.Timedelta(days=1)

starts = anti[['Patient_ID','start_day']].rename(columns={'start_day':'Day'}).assign(delta=1)
ends   = anti[['Patient_ID','end_day_plus1']].rename(columns={'end_day_plus1':'Day'}).assign(delta=-1)
deltas = pd.concat([starts, ends], ignore_index=True)
deltas = deltas.groupby(['Patient_ID','Day'], as_index=False)['delta'].sum()

timeline = timeline.merge(deltas, on=['Patient_ID','Day'], how='left')
timeline['delta'] = timeline['delta'].fillna(0).astype('int16')
timeline['anti_running_sum'] = timeline.groupby('Patient_ID')['delta'].cumsum()
timeline['on_antipsychotic'] = (timeline['anti_running_sum'] > 0).astype('int8')
timeline.drop(columns=['delta','anti_running_sum'], inplace=True)

# PRN starts
is_prn = df_c['Prescription_Type'].str.contains('PRN', case=False, na=False)
prn = (df_c.loc[is_prn, ['Patient_ID','start_day']]
       .rename(columns={'start_day':'Day'}))
prn_counts = (prn.groupby(['Patient_ID','Day']).size()
              .rename('prn_starts').reset_index())
timeline = timeline.merge(prn_counts, on=['Patient_ID','Day'], how='left')
timeline['prn_starts'] = timeline['prn_starts'].fillna(0).astype('int16')

timeline.sort_values(['Patient_ID','Day'], inplace=True, ignore_index=True)
timeline['num_prn_72h'] = (timeline.groupby('Patient_ID', group_keys=False)['prn_starts']
                           .rolling(window=3, min_periods=1).sum()
                           .reset_index(level=0, drop=True).astype('int16'))

# Demographics
demo_cols = ['Patient_ID','Age','Gender','Diagnosis','MH_Section']
demo = df_a[demo_cols].drop_duplicates('Patient_ID')
demo['MH_Section'] = demo['MH_Section'].fillna('Unknown')
timeline = timeline.merge(demo, on='Patient_ID', how='left')

# Gender dummies
gender_dum = pd.get_dummies(timeline['Gender'], prefix='Gender', dtype='int8')
timeline = pd.concat([timeline, gender_dum], axis=1)

# Label: aggression next 24h
timeline['y_next24h'] = (timeline.groupby('Patient_ID')['aggression_today']
                         .shift(-1).fillna(0).astype('int8'))

# Final feature list
feature_cols = [
    'on_antipsychotic',
    'num_prn_72h',
    'activity_today',
    'time_since_last_activity_hours',
    'Age'
] + [c for c in timeline.columns if c.startswith('Gender_')] + [
    'aggression_today',
    'any_restraint',
    'any_selfharm',
    'any_safeguarding',
    'any_obs_event',
    'restraint_physical_aggression'
]

X_frame = timeline[['Patient_ID','Day'] + feature_cols].copy()
y_frame = timeline[['Patient_ID','Day','y_next24h']].copy()

# Sequence building
patient_sequences = []
patient_labels = []
for pid, g in X_frame.groupby('Patient_ID'):
    g = g.sort_values('Day')
    X = g[feature_cols].to_numpy(dtype='float32')
    y = y_frame.loc[g.index, 'y_next24h'].to_numpy(dtype='float32')
    patient_sequences.append(X)
    patient_labels.append(y)

X_padded = pad_sequences(patient_sequences, padding='post', dtype='float32')
y_padded = pad_sequences(patient_labels, padding='post', dtype='float32')

print("X_padded:", X_padded.shape, "| y_padded:", y_padded.shape)

# Save
timeline.to_csv("enriched_timeline.csv", index=False)
np.save("X_padded.npy", X_padded)
np.save("y_padded.npy", y_padded)